<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c34-header.png" alt="Nextia Learning · Anomaly Detection: Find What Is Unusual" width="100%">

# Solution: Has the distribution changed?

**[Has the distribution changed?](https://learning.nextia-ai.com/courses/anomaly-detection/m03/has-the-distribution-changed/)** · Anomaly Detection: Find What Is Unusual · Module 3, lesson 5 of 7 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** compare a reference window with a current one: the Kolmogorov-Smirnov test and the Wasserstein distance for numbers, the chi-square test for categories, and the Population Stability Index (PSI). You will see why a big sample says "changed" for a tiny change, and why the size of a change answers the business question.

| | |
|---|---|
| **Time** | About 40 minutes |
| **Needs** | An internet connection for the setup cells (about 7 MB). No account. |
| **You should know** | A p-value and a median, from [Essential Mathematics and Statistics for AI](https://learning.nextia-ai.com/courses/math/); the payments table, from [Meet the Larkfield data](https://learning.nextia-ai.com/courses/anomaly-detection/m01/meet-the-larkfield-data/). |
| **You do not need** | A drift tool: every test is one SciPy call. |
| **Tested** | Python 3.12.13 in Jupyter on macOS (Apple silicon), from a fresh start (*Restart and run all*), with the package versions of the setup cell |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/anomaly-detection/m03/has-the-distribution-changed/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C34/notebooks/M03-L05-has-the-distribution-changed/has-the-distribution-changed-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs the versions that the course uses: pandas 3.1.1, numpy 2.5.4, scipy 1.18.1, scikit-learn 1.9.1, matplotlib 3.11.2. It needs an internet connection. The first time, it takes up to a minute.

You should see one line per package, with the same versions. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again. A restart is needed when an older version was already loaded.

In [ ]:
%pip install -q --disable-pip-version-check pandas==3.1.1 numpy==2.5.4 scipy==1.18.1 scikit-learn==1.9.1 matplotlib==3.11.2
from importlib.metadata import version
for p in ['pandas', 'numpy', 'scipy', 'scikit-learn', 'matplotlib']:
    print(p, version(p))

## Setup: get the course data

Run the next cell. It downloads the data that this lesson uses (`payments.csv.gz`, `sealed-key.txt`) into a folder `anomaly-course/data`, and checks each file's checksum. It also downloads `compare.py`, the course's shared code. It needs an internet connection and no account. It takes a few seconds.

You should see `Ready:` and the files with their sizes. If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `anomaly-course` (in Colab: the **Files** panel on the left), then run the cell again.

The data is made up for this course: no real people, cards or payments. [The dataset card](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C34/data/dataset-card.md) describes every table.

In [ ]:
# Setup: download the course data into anomaly-course/data/ and check each file.
# Standard library only.
import hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

LABS = "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C34/"
FILES = {
    "payments.csv.gz": "c4e18c1bbaf422cfc99821f2a1b65cf0e8274f835c40ca757a37a531eeccd542",
    "sealed-key.txt": "144cfd16737bdfcc4e992ad8ede0b8856265a2193c89fa7266fe0b27cb62ff98",
}
CODE_FILES = ["compare.py"]
HERE = Path.cwd()
# The notebook works in its own folder anomaly-course/, marked with a file, so it never writes into another project.
if Path(".nextia-notebook").exists():          # the cell ran before: start again from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
WORK = Path("anomaly-course").resolve()
(WORK / "data").mkdir(parents=True, exist_ok=True)
(WORK / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

def fetch(name, url, path, expected=None):
    """Use a copy next to the notebook if there is one, else download it."""
    if path.exists() and (expected is None or sha256(path) == expected):
        return
    if (HERE / name).exists():
        shutil.copy(HERE / name, path)
    else:
        download(url, path)
    if expected and sha256(path) != expected:
        raise SystemExit(f"{name} has the wrong checksum. Delete the folder anomaly-course and run again.")

for name, expected in FILES.items():
    fetch(name, LABS + "data/" + name, WORK / "data" / name, expected)
for name in CODE_FILES:
    fetch(name, LABS + name, WORK / name)
os.chdir(WORK)
print("Ready:", ", ".join(f"{n} ({(Path('data') / n).stat().st_size / 1e6:.1f} MB)" for n in FILES))

## Setup: helpers

The next cell defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

It also loads pandas as `pd`, NumPy as `np` and matplotlib as `plt`. Run the cell. It prints nothing.

In [ ]:
# Helpers: check an answer without showing it, and the chart style.
import hashlib, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
pd.set_option("display.precision", 3)
pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 20)
warnings.filterwarnings("ignore", category=FutureWarning)
plt.rcParams.update({"figure.figsize": (9, 3.2), "axes.spines.top": False, "axes.spines.right": False})

def fingerprint(value):
    def norm(v):
        if hasattr(v, "to_dict") and hasattr(v, "columns"):
            return {"columns": [str(c) for c in v.columns], "rows": [norm(list(r)) for r in v.itertuples(index=False)]}
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple, set)):
            return [norm(x) for x in (sorted(v, key=str) if isinstance(v, set) else v)]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 4)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. Two small samples: KS and Wasserstein

The **Kolmogorov-Smirnov (KS) test** compares two samples of numbers. For each value, it takes the share of each sample at or below it (the **empirical cumulative distribution**). The KS statistic is the largest gap between the two shares. The **Wasserstein distance** is how far, on average, each value must move to turn one sample into the other, in the units of the data.

> **Predict.** Last month: 10, 20, 30, 40, 50 and 60 dollars. This month: 15, 25, 35, 45, 55 and 95. Which measure sees the 95 most clearly?

> **Check.** You should see a KS statistic of 0.167 and a Wasserstein distance of 10.0.

In [ ]:
from scipy import stats
ref = np.array([10, 20, 30, 40, 50, 60])              # six amounts last month
cur = np.array([15, 25, 35, 45, 55, 95])              # six amounts this month
grid = np.sort(np.concatenate([ref, cur]))
ecdf = pd.DataFrame({"amount": grid,
                     "share of ref at or below": [(ref <= v).mean() for v in grid],
                     "share of cur at or below": [(cur <= v).mean() for v in grid]})
ecdf["gap"] = (ecdf.iloc[:, 1] - ecdf.iloc[:, 2]).abs()
print(ecdf.round(3).to_string(index=False))
ks = stats.ks_2samp(ref, cur)
print(f"KS statistic {ks.statistic:.3f} (the largest gap), p-value {ks.pvalue:.3f}")
print("Wasserstein distance:", stats.wasserstein_distance(ref, cur), "= the mean of", np.abs(np.sort(cur) - np.sort(ref)).tolist())

The shares never differ by more than one value in six, so KS sees almost nothing (p = 1.000). The Wasserstein distance is 10 dollars: five values moved 5 dollars, and one moved 35.

## 2. PSI on four bins

The **Population Stability Index (PSI)** comes from credit risk. Cut the reference into bins, compare the share of each bin, and add up (cur − ref) × ln(cur / ref). Each term is positive, so changes never cancel.

> **Check.** You should see a PSI of 0.0203.

In [ ]:
bins = pd.DataFrame({"ref share": [0.25, 0.25, 0.25, 0.25], "cur share": [0.20, 0.25, 0.25, 0.30]},
                    index=["under 20", "20 to 45", "45 to 100", "100 and more"])
bins["(cur - ref) x ln(cur / ref)"] = (bins["cur share"] - bins["ref share"]) * np.log(bins["cur share"] / bins["ref share"])
print(bins.round(4).to_string())
print("PSI:", round(bins.iloc[:, 2].sum(), 4))

Common rules of thumb: below 0.1, no real change; 0.1 to 0.25, a moderate change; above 0.25, a large change. These limits are **conventions** from credit scoring, not laws: choose your own for your data and your decision.

Run the next cell. It defines `psi()`, `js()` (the Jensen-Shannon distance, another measure on the same bins), `compare_numbers()` and `psi_categories()`. It prints nothing.

In [ ]:
def psi(ref, cur, bins=10):
    """Population Stability Index, with 10 bins from the quantiles of the reference."""
    edges = np.unique(np.quantile(ref, np.linspace(0, 1, bins + 1)))
    edges[0], edges[-1] = -np.inf, np.inf
    r = np.clip(np.histogram(ref, edges)[0] / len(ref), 1e-6, None)
    c = np.clip(np.histogram(cur, edges)[0] / len(cur), 1e-6, None)
    return float(np.sum((c - r) * np.log(c / r)))

def js(ref, cur, bins=10):
    """Jensen-Shannon distance on the same bins (0 = the same, 1 = nothing in common)."""
    from scipy.spatial.distance import jensenshannon
    edges = np.unique(np.quantile(ref, np.linspace(0, 1, bins + 1)))
    edges[0], edges[-1] = -np.inf, np.inf
    return float(jensenshannon(np.histogram(ref, edges)[0], np.histogram(cur, edges)[0], base=2))

def compare_numbers(ref, cur):
    ks = stats.ks_2samp(ref, cur)
    return {"n ref": len(ref), "n cur": len(cur), "KS statistic": round(float(ks.statistic), 4), "KS p": float(f"{ks.pvalue:.3g}"),
            "Wasserstein": round(float(stats.wasserstein_distance(ref, cur)), 3), "PSI": round(psi(ref, cur), 4),
            "JS distance": round(js(ref, cur), 4), "median ref": round(float(np.median(ref)), 2),
            "median cur": round(float(np.median(cur)), 2)}

def psi_categories(ref, cur):
    cats = sorted(set(ref) | set(cur))
    r = np.clip(pd.Series(ref).value_counts(normalize=True).reindex(cats, fill_value=0), 1e-6, None)
    c = np.clip(pd.Series(cur).value_counts(normalize=True).reindex(cats, fill_value=0), 1e-6, None)
    return float(np.sum((c - r) * np.log(c / r)))

## 3. The new region and the payment amounts

The east region went live on 2025-09-01 (an anomaly we planted: a normal change for good). We compare August (the reference) with 2025-09-15 to 2025-10-14, after the start-up weeks.

In [ ]:
payments = pd.read_csv("data/payments.csv.gz", parse_dates=["ts"])
day = payments.ts.dt.floor("D")
august = payments[(day >= "2025-08-01") & (day <= "2025-08-31")]           # the reference window
after = payments[(day >= "2025-09-15") & (day <= "2025-10-14")]            # after the east region went live
july = payments[(day >= "2025-07-01") & (day <= "2025-07-31")]
print(f"August {len(august):,} payments; 2025-09-15 to 2025-10-14 {len(after):,}; east in the new window {(after.region == 'east').mean():.1%}")

> **Predict.** About one payment in nine now comes from the new region. Will KS say "changed"? Will PSI be above 0.1?

> **Check.** You should see `KS p` 7.24e-29 and `PSI` 0.0068.

In [ ]:
amount = compare_numbers(august.amount.to_numpy(), after.amount.to_numpy())
for name, value in amount.items():
    print(f"{name:>13}: {value}")

KS says "changed" with a p-value of 7.24e-29. PSI says 0.0068: far below 0.1. The median fell from 45.12 to 41.9 dollars. Both are right. The change is real, and it is small. Run the next cell to see the two shapes.

In [ ]:
edges = np.geomspace(1, 5000, 40)
fig, ax = plt.subplots(figsize=(9, 3.2))
ax.hist(august.amount, bins=edges, density=True, histtype="step", lw=1.4, label="August (reference)")
ax.hist(after.amount, bins=edges, density=True, histtype="step", lw=1.4, label="2025-09-15 to 10-14")
ax.set_xscale("log")
ax.set_xlabel("amount (dollars, log scale)")
ax.legend()
plt.show()

In [ ]:
old = after[after.region != "east"]
without_east = compare_numbers(august.amount.to_numpy(), old.amount.to_numpy())
print("without the east region:", {k: without_east[k] for k in ("KS p", "PSI", "median cur")})
print("east region median amount:", after[after.region == "east"].amount.median())

Without the east region, the old regions changed very little (PSI 0.0008). The new region has smaller baskets, and it is one ninth of the payments.

## 4. Categories: chi-square and PSI

For categories, the **chi-square test** compares the counts of each category in the two windows. PSI works on the shares of the categories.

In [ ]:
for feature in ["region", "device_type"]:
    cats = sorted(set(august[feature]) | set(after[feature]))
    counts = np.vstack([august[feature].value_counts().reindex(cats, fill_value=0),
                        after[feature].value_counts().reindex(cats, fill_value=0)])
    chi = stats.chi2_contingency(counts)
    print(f"{feature}: chi-square {chi.statistic:.1f}, p {chi.pvalue:.3g}, PSI {psi_categories(august[feature], after[feature]):.4f}")
    print(pd.DataFrame(counts / counts.sum(axis=1, keepdims=True), index=["August", "after"], columns=cats).round(3).to_string())

The region mix changed a lot (PSI above 1): a new category appeared. The device mix also gives a tiny p-value, with a PSI near 0.005.

## 5. Why a big sample says "changed"

> **Predict.** We take the same two windows, but only n payments from each. As n grows from 200 to 50,000, what happens to the KS p-value? And to the PSI?

In [ ]:
rng = np.random.default_rng(7)
rows = []
for n in (200, 1000, 5000, 20000, 50000):
    draws = [compare_numbers(rng.choice(august.amount.to_numpy(), n, replace=False),
                             rng.choice(after.amount.to_numpy(), n, replace=False)) for _ in range(20)]
    rows.append({"payments in each window": n,
                 "KS p (median of 20 draws)": float(f"{np.median([d['KS p'] for d in draws]):.3g}"),
                 "KS statistic": round(float(np.median([d["KS statistic"] for d in draws])), 3),
                 "PSI": round(float(np.median([d["PSI"] for d in draws])), 4)})
sizes = pd.DataFrame(rows)
print(sizes.to_string(index=False))

> **Check.** You should see a median p-value of 0.713 with 200 payments and 4.29e-26 with 50,000.

The change is the same in every row. Only the sample grows. The p-value falls toward 0, because a test with more data can see a smaller change. The KS statistic and the PSI settle near the size of the change. With small samples, PSI is noisy and too high. A p-value answers "is there any change at all?" The size of the change answers the business question: "is it big enough to act on?"

## 6. A harmless change

Run the next cell. It compares the hour of the payments.

In [ ]:
hour = compare_numbers(august.hour.to_numpy(), after.hour.to_numpy())
print({k: hour[k] for k in ("KS p", "Wasserstein", "PSI")})
print(f"payments came {hour['Wasserstein'] * 60:.0f} minutes later on average, by the Wasserstein distance")

A p-value of 6.97e-09, for payments that come about 12 minutes later. No one needs to act on that.

## 7. Check yourself

**Your turn 1.** Four bins, reference shares 25% each, current shares 10%, 20%, 30% and 40%. What is the PSI? Round to 4 decimals and write it as `psi_tiny_2`.

In [ ]:
bins2 = pd.DataFrame({"ref": [0.25, 0.25, 0.25, 0.25], "cur": [0.10, 0.20, 0.30, 0.40]})
psi_tiny_2 = round(float(((bins2.cur - bins2.ref) * np.log(bins2.cur / bins2.ref)).sum()), 4)

In [ ]:
expect('psi_tiny_2', psi_tiny_2, 0.2282, tolerance=0.0001)

**Your turn 2.** Compare the amounts of the full months of July and August with `stats.ks_2samp`. Write the p-value, rounded to 3 decimals, as `july_august_p`.

In [ ]:
july_august_p = round(stats.ks_2samp(july.amount, august.amount).pvalue, 3)

In [ ]:
expect('july_august_p', july_august_p, 0.654, tolerance=0.001)

**Your turn 3.** What is the PSI of the `currency` column between August and the window after the new region? Use `psi_categories`, round to 4 decimals, and write it as `currency_psi`.

In [ ]:
currency_psi = round(psi_categories(august.currency, after.currency), 4)

In [ ]:
expect('currency_psi', currency_psi, 0.0064, tolerance=0.0001)

## Recap

- KS and Wasserstein compare numbers; chi-square compares categories; PSI and the Jensen-Shannon distance measure the size of a change on bins.
- The new region moved the amounts: KS p 7.24e-29, PSI 0.0068, median 45.12 → 41.9 dollars. Real, and small.
- With more data, the p-value of the same change falls toward 0. Decide with the size of the change and a limit that you chose for your decision.

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Drift in many features, and what to do about it](https://learning.nextia-ai.com/courses/anomaly-detection/m03/drift-in-many-features/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/anomaly-detection/m03/has-the-distribution-changed/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The practice data is synthetic and has no real people.